In [ ]:
from ase.io import read, write
from ase.visualize import view
from rdkit.Chem import Draw

import os, yaml, glob

from openmm.openmm import XmlSerializer
from openmm.app import PME
from openmm.app.forcefield import ForceField
from openmm.app.pdbfile import PDBFile

from imolcryff.asemol import asemol_wrapper, aseatoms2pdb, merge_asemols
from imolcryff.molinfo import Mol_Info
from imolcryff.g16wrapper import molinfo_setg16opt, molinfo_setg16charge, molinfo_setg16dihedral
from imolcryff.gaffil_generators import GAFFilTemplateGenerator
from imolcryff.ffxml import gafftemplate2xml

In [ ]:
import imolcryff
imolcryff.__file__

In [ ]:
params = yaml.safe_load(open('all_params.yaml'))
params

In [ ]:
atoms = read("ordered_Li(FSA)(C2)2_233K.cif")
asemol_wrap = asemol_wrapper(atoms)
atoms_unwrap = asemol_wrap.unwrap_molecules()
atomslist_mols, molecule_list = asemol_wrap.get_ase_molecules(ordered=True)

merged_aseatoms = merge_asemols(atomslist_mols)
aseatoms2pdb("merged.pdb", merged_aseatoms)

In [ ]:
import rdkit
rdkit.__version__

In [ ]:
mol_info = Mol_Info()

In [ ]:
for i, mol_idx in enumerate(molecule_list):
    al = [atomslist_mols[i] for i in mol_idx]
    mol_info.append_fromAtomsList(al, key=f"MOL_{i}", Nmols = len(mol_idx))

In [ ]:
mol_info.auto_charge_assign()
mol_info.get_rdkitmol()
mol_info.il_assign()
rdkitmol_list = [mol_info.mol_info[key]["rdkitmol2d"] for key in mol_info.mol_info.keys()]
Draw.MolsToGridImage(rdkitmol_list, molsPerRow = 3, subImgSize = (300,150),
                     legends = [molkey for molkey in mol_info.mol_info.keys()], maxMols = 60)

In [ ]:
default_params_dihedral = {
    "method": "wb97xd",
    "basis": "6-311++g(d,p)",
    "opt": "modredundant",
    "mem": "92GB",
    "nprocshared": 40,
    "addsec": "hoge-hoge"
}

In [ ]:
mol_info.get_smiles_from_molinfo()
mol_info.get_sdf_from_molinfo()
molinfo_setg16opt(mol_info.mol_info, params["g16opt"])
mol_info.do_geoopt()  # you need Gaussian16
mol_info.load_geoopt_results()

molinfo_setg16charge(mol_info.mol_info, params["charge"])
mol_info.do_charge()

# if you want to calculate dihedral potential energy profile
# mol_info.get_rotatable_dihedral()
# molinfo_setg16dihedral(mol_info.mol_info)
# mol_info.do_dihedral()   # you need Gaussian16
# mol_info.load_dihedral_results()

In [ ]:
for i in range(10):
    try:
        _ = mol_info.get_charges_from_molinfo()
    except:
        pass

In [ ]:
mol_info.adjust_charges(params["ff_params"]) # necesarry even for neutral species

In [ ]:
mol_info.save_molinfo("molinfo.pkl")

In [ ]:
mmm = mol_info.get_molecule_omm()
if params["ff_params"]["fftype"].split("-")[0] == "gaff":
    gaff = GAFFilTemplateGenerator(molecules=mmm, forcefield=params["ff_params"]["fftype"], il_assign=params["fsa_assign"])

gafftemplate2xml(mmm, gaff, ion_ffxml=params["ff_params"]["iontype"])

In [ ]:
atomspdb = read("merged.pdb") 
atomspdb = atomspdb.repeat((1,1,1))
aseatoms2pdb("merged_supercell.pdb", atomspdb)

pdb = PDBFile("merged_supercell.pdb")
pdb_b = asemol_wrapper(atomspdb)
_ = pdb_b.get_bonds()
bonds_list = [ [bond[0], bond[1]] for bond in pdb_b.bonds]
atomlist_openmm = [a for a in pdb.topology.atoms()]
for bond in bonds_list:
    a1 = atomlist_openmm[bond[0]]
    a2 = atomlist_openmm[bond[1]]
    pdb.topology.addBond(a1, a2)

pdb.writeFile(pdb.topology, pdb.positions, open("merged_supercell_bonds.pdb", "w"))
pdb = PDBFile("merged_supercell_bonds.pdb")

In [ ]:
gaff_files = sorted(glob.glob("gaffxml_*.xml"))
forcefield = ForceField(*gaff_files)
sys = forcefield.createSystem(pdb.topology, nonbondedMethod=PME)
with open('system.xml', 'w') as output:
    output.write(XmlSerializer.serialize(sys))

In [ ]:
from openmm import app
from parmed.openmm import load_topology

parm_top = load_topology(topology=pdb.topology, system=sys, xyz=pdb.positions)

parm_top.save('test.top', overwrite=True)
parm_top.save('test.gro', overwrite=True)

In [ ]:
from openff.interchange.components.mdconfig import MDConfig
from openff.toolkit import Topology
from openff.interchange.interop import openmm
from openff.interchange.interop.lammps.export import to_lammps
from imolcryff.exporter_lmp import to_lammps_non_rectangular

topology_off = Topology.from_openmm(pdb.topology, unique_molecules=mmm, positions=pdb.getPositions())
os.environ["INTERCHANGE_EXPERIMENTAL"] = "1"
a = openmm.from_openmm(system=sys, topology=topology_off, positions=pdb.getPositions())
to_lammps(a, "test.data")
# non-rectangular box
# to_lammps_non_rectangular(a, "test.data")